In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, Subset
from sklearn.model_selection import train_test_split

In [4]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness = 0.2, contrast = 0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

test_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])

In [5]:
dataset = datasets.ImageFolder("color/")

In [6]:
dataset.class_to_idx

{'Apple___Apple_scab': 0,
 'Apple___Black_rot': 1,
 'Apple___Cedar_apple_rust': 2,
 'Apple___healthy': 3,
 'Blueberry___healthy': 4,
 'Cherry_(including_sour)___Powdery_mildew': 5,
 'Cherry_(including_sour)___healthy': 6,
 'Corn_(maize)___Cercospora_leaf_spot Gray_leaf_spot': 7,
 'Corn_(maize)___Common_rust_': 8,
 'Corn_(maize)___Northern_Leaf_Blight': 9,
 'Grape___Black_rot': 10,
 'Grape___Esca_(Black_Measles)': 11,
 'Grape___Leaf_blight_(Isariopsis_Leaf_Spot)': 12,
 'Grape___healthy': 13,
 'Orange___Haunglongbing_(Citrus_greening)': 14,
 'Peach___Bacterial_spot': 15,
 'Peach___healthy': 16,
 'Pepper_bell___Bacterial_spot': 17,
 'Pepper_bell___healthy': 18,
 'Potato___Early_blight': 19,
 'Potato___Late_blight': 20,
 'Potato___healthy': 21,
 'Raspberry___healthy': 22,
 'Soybean___healthy': 23,
 'Squash___Powdery_mildew': 24,
 'Strawberry___Leaf_scorch': 25,
 'Strawberry___healthy': 26,
 'Tomato___Bacterial_spot': 27,
 'Tomato___Early_blight': 28,
 'Tomato___Late_blight': 29,
 'Tomato__

In [7]:
x = list(range(len(dataset)))
y = [label for _, label in dataset.samples]

train_idx, test_idx = train_test_split(
    x, test_size = 0.2, stratify = y, random_state = 42
)

In [8]:
train_dataset = datasets.ImageFolder("color/", transform = train_transform)
test_dataset = datasets.ImageFolder("color/", transform = test_transform)

train_data = Subset(train_dataset, train_idx)
test_data = Subset(test_dataset, test_idx)

In [9]:
trainloader = DataLoader(train_data, batch_size=64, shuffle=True)
testloader  = DataLoader(test_data, batch_size=64, shuffle=False)

## Build CNN

In [10]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3,32, kernel_size = 3, padding =1),
            nn.ReLU(),
            nn.MaxPool2d(2,2), ## kernel size = 2, stride = 2

            nn.Conv2d(32,64, kernel_size = 3, padding =1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),

            nn.Conv2d(64,128, kernel_size = 3, padding =1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),       
        )

        self.fc_layers = nn.Sequential(
            nn.Linear(28*28*128, 256),
            nn.ReLU(),

            nn.Linear(256,37)
        )

    def forward(self, x):
        x = self.conv_layers(x)
        x = x.view(x.size(0), -1) # # flattening
        x = self.fc_layers(x)

        return x

In [11]:
model = CNN()

In [12]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

## Train CNN

In [13]:
epochs = 10

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()
        
        output = model.forward(images) # FP
        loss = criterion(output, labels) # loss fnx
        loss.backward() # BP
        optimizer.step() # update params
        epoch_training_loss += loss.item()

    print(f"epoch={epoch+1}/{epochs} & loss={epoch_training_loss/len(trainloader)}")

epoch=1/10 & loss=1.707203467537586
epoch=2/10 & loss=0.9985285557302317
epoch=3/10 & loss=0.7722748855899151
epoch=4/10 & loss=0.6497585056867815
epoch=5/10 & loss=0.5816967826140554
epoch=6/10 & loss=0.5198339893629677
epoch=7/10 & loss=0.4830648156933318
epoch=8/10 & loss=0.45901682502345037
epoch=9/10 & loss=0.4290287733750236
epoch=10/10 & loss=0.4257282176300099


In [16]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in testloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"accuracy = {correct_labels / total_labels * 100}")

accuracy = 93.78116473798099


In [17]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images, labels in trainloader:
        outputs = model.forward(images)
        _, predicted  = torch.max(outputs, 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Train Accuracy = {correct_labels / total_labels * 100}")

Train Accuracy = 87.5617443665616


In [14]:
torch.save(model.state_dict(), "model.pth")

In [15]:
model.load_state_dict(torch.load("model.pth"))
model.eval()

CNN(
  (conv_layers): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Linear(in_features=100352, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=37, bias=True)
  )
)